In [17]:
import kagglehub
import json
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder,
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import f1_score

In [18]:
with open("../kaggle_token.json", "r") as f:
    kaggle_acc = json.load(f)

os.environ["KAGGLE_USERNAME"] = kaggle_acc["username"]
os.environ["KAGGLE_KEY"] = kaggle_acc["key"]

path = kagglehub.competition_download('ud-s-machine-learning-2026-classification')

print("Path to competition files:", path)

Path to competition files: C:\Users\glebo\.cache\kagglehub\competitions\ud-s-machine-learning-2026-classification


In [19]:
train = pd.read_csv(path+"/train.csv")

TARGET = "Demand_Category"

train = train.dropna()

X = train.drop(columns=[TARGET, "Date"])
y_raw = train[TARGET]

train.head(2)


,Kaggle_ID,Date,Hour,Temperature,Humidity,Wind speed,Visibility,Dew point temperature,Solar Radiation,Rainfall,Snowfall,Seasons,Holiday,Functioning Day,Record_id,Demand_Category
0,0,15/05/2018,11,24.8,36,1.8,973,8.7,2.7,0.0,0.0,Sprng,No Holiday,Yes,5137441,1
1,1,7/9/2018,23,12.9,0,2.4,1688,50.9,0.0,2.4,1.4,Sproing,No Holiday,No,1465289,0


In [20]:
# =========================
# 2. ENCODE TARGET
# =========================
le = LabelEncoder()

y = le.fit_transform(y_raw)


# =========================
# 3. COLUMN SPLIT
# =========================
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


# =========================
# 4. PREPROCESSOR
# =========================
numeric_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)


# =========================
# 5. PIPELINE
# =========================
pipeline = Pipeline(
    [
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=2000)),
    ]
)


# =========================
# 6. CROSS VALIDATION
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipeline,
    X,
    y,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
)

print("\nCV Macro F1:", scores.mean())


# =========================
# 7. FINAL MODEL
# =========================
pipeline.fit(X, y)

final_pipeline = pipeline

print("\nModel trained successfully")

C:\Users\glebo\AppData\Local\Temp\ipykernel_17208\3192926253.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()



CV Macro F1: 0.9307320503042756

Model trained successfully


# Submission

In [21]:
# %%
import pandas as pd


# =========================
# 8. TEST PREDICTIONS
# =========================
test = pd.read_csv(path+"/test.csv")

test_X = test.drop(columns=["Date"])

preds = final_pipeline.predict(test_X)

# convert labels back
pred_labels = le.inverse_transform(preds)


# =========================
# 9. SUBMISSION
# =========================
submission = pd.read_csv(path+"/sample_submission.csv")

submission["Demand_Category"] = pred_labels

submission.to_csv("my_first_submission.csv", index=False)

print("submission created successfully")
print(submission.head())

submission created successfully
   Kaggle_ID  Demand_Category
0          0                0
1          1                0
2          2                0
3          3                0
4          4                0
